In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../../scripts')

In [3]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = '/data2/a330d' #os.environ.get("DATA_ROOT", ".")
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd

from scipy.stats import pearsonr, spearmanr

from cellina import make_neighbor_perturbation
from cellina_graph import make_perturbed_expression
from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, split_indices, preprocess_spatial_features
from counterfactual_analysis import compute_rmse, compute_edistance, mixing_index, get_lfc, precision, direction_match, compute_mse_lfc, _to_dense
from counterfactual_analysis import get_perturbation_logfc, get_global_perturbation_logfc
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import cellina

cellina.__version__

'0.7.4'

In [5]:
set_seed(0)

In [6]:
DATASET_NAME = "crc"  # or "merfish"
CELLINA_BASE_MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained")
CELLINA_GAT_MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained")

In [7]:
CRC_PATHS = [
    #os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_231.h5ad"),
    #os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_232.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_242.h5ad"),
]

CRC_HOLDOUTS = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_PATHS = [
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad"),    
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad"),
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad"),
]

MERFISH_HOLDOUTS = [
    'glutamatergic neuron',
    'oligodendrocyte',
    'astrocyte',
    'GABAergic neuron',
    'endothelial cell',
]

PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
HOLDOUT_CELLTYPES = CRC_HOLDOUTS if DATASET_NAME == "crc" else MERFISH_HOLDOUTS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH
COUNTS_PER_K = 1e4

In [8]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 512
library_size = 'latent'
n_deg = 50
n_pert_genes = 200

In [9]:
# Create SLIDES which contain file names from PATHS - first split by "/" and take last part, then split by "." and take first part
SLIDES = [path.split("/")[-1].split(".h5ad")[0] for path in PATHS]

In [10]:
results = []
model_names = ['cellina-W_1'] #['cellina-W']#, 

for path, slide_id in zip(PATHS, SLIDES):
    adata = sc.read(path)
    
    if DATASET_NAME == 'crc':
        adata = preprocess_crc(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    elif DATASET_NAME == 'merfish':
        adata = preprocess_merfish(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    else:
        raise ValueError(f"Unknown dataset_name: {DATASET_NAME}. Supported: crc, merfish")
    
    for holdout_celltype in HOLDOUT_CELLTYPES:
        # 50 times * in print
        print(f"{'='*50} Slide: {slide_id}, Holdout Celltype: {holdout_celltype} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_celltype,
                                                    labels_key=labels_key,
                                                    domains_key=domains_key,
                                                    holdout_domains=holdout_domains,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)
        # Compute spatial features after splitting to avoid data leakage
        step_size_px = 0.12028 if DATASET_NAME == 'crc' else 0.109
        adata = preprocess_spatial_features(adata, step_size_px=step_size_px, n_neighbors=n_neighbors, test_indices=test_idx)
        
        for model_name in model_names:
            if model_name == 'cellina-W_1':
                 save_name = "cellina"
                 model_class = "cellina"
            else:
                save_name = "cellina-gat-pert"
                model_class = "cellina_graph"
            
                        
            MODEL_ROOT = CELLINA_BASE_MODEL_ROOT if model_class == 'cellina' else CELLINA_GAT_MODEL_ROOT
            save_dir = os.path.join(MODEL_ROOT, slide_id, holdout_celltype, model_name)
            
            try:
                model = _load_model(save_dir,
                                    model_class=model_class,
                                    adata=adata,
                                    splits=splits)
            except Exception as e:
                print(f"Failed to load model from {save_dir} with error: {e}")
                continue
            is_control_region = adata.obs[domains_key]==(control_domain)
            is_holdout_ct = adata.obs[labels_key].astype(str) == holdout_celltype
            mask_control = is_control_region & is_holdout_ct
            idx_control = np.where(mask_control.values)[0]    
            
            for hd in holdout_domains:                
                is_holdout_region = adata.obs[domains_key].astype(str) == hd
                mask_target = is_holdout_region & is_holdout_ct
                idx_target = np.where(mask_target.values)[0]

                # Compute stats (control/target populations are the same across neighbor_ct choices)
                control = adata.layers['counts'][mask_control.values, :]
                target = adata.layers['counts'][mask_target.values, :]
                control, target = _to_dense(control), _to_dense(target)

                for neighbor_ct in HOLDOUT_CELLTYPES:
                    # "neighbour_indices" are cells of neighbor_ct within the holdout (target) domain hd
                    is_neighbor_ct = adata.obs[labels_key].astype(str) == neighbor_ct
                    mask_neighbor = is_holdout_region & is_neighbor_ct
                    neighbor_indices = np.where(mask_neighbor.values)[0]

                    if len(neighbor_indices) == 0:
                        print(f"Skipping {holdout_celltype}/{hd}/neighbor_ct={neighbor_ct}: no neighbor cells found")
                        continue

                    args_gex = {
                        "indices": idx_control,
                        "batch_size": batch_size,
                        "seed": 0,
                        "neighbour_indices": neighbor_indices
                    }
                    if model_class.lower() == 'cellina_graph':
                        args_gex["n_neighbors_per_seed"] = 50
                    else:
                        args_gex['precomputed'] = False

                    cf_counts = model.get_counterfactual_expression(**args_gex)
                    counterfactual = cf_counts

                    gt_lfc, cf_lfc, deg = get_lfc(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg)

                    spear, _ = spearmanr(gt_lfc[deg], cf_lfc[deg])
                    pear, _ = pearsonr(gt_lfc[deg], cf_lfc[deg])
                    prec = precision(gt_lfc, cf_lfc, k=n_deg, use_abs=True)
                    dir_match = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="intersection")
                    dir_match_k = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="k")
                    dir_match_gt = direction_match(gt_lfc, cf_lfc, k=n_deg, normalize="gt_topk")
                    mix_idx = mixing_index(observed=target, predicted=counterfactual, library_size=COUNTS_PER_K)
                    edist_global = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K)
                    edist_local = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True)
                    edist_pca_log = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True)
                    edist_pca = compute_edistance(adata, observed=target, predicted=counterfactual, deg=None, library_size=COUNTS_PER_K, local=True, use_pca=True, log1p=False)
                    rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg, library_size=COUNTS_PER_K)
                    mse_lfc = compute_mse_lfc(gt_vec=gt_lfc, cf_vec=cf_lfc, deg=deg)

                    results.append(
                            dict(
                            dataset_name=DATASET_NAME,
                            sid=slide_id,
                            control_domain=control_domain,
                            target_domain=hd,
                            n_deg=n_deg,
                            model_name=save_name,
                            holdout_celltype=holdout_celltype,
                            neighbor_celltype=neighbor_ct,
                            spearman=spear,
                            pearson=pear,
                            precision=prec,
                            direction_match=dir_match,
                            direction_match_k=dir_match_k,
                            direction_match_gt=dir_match_gt,
                            mixing_index=mix_idx,
                            edistance_global=edist_global,
                            edistance_local=edist_local,
                            edistance_pca_log=edist_pca_log,
                            edistance_pca=edist_pca,
                            rmse=rmse,
                            mse_lfc=mse_lfc,
                            top_n_perturb=n_pert_genes,
                            )
                    )

/data/a330d/projects/cellina-reproducibility/notebooks/loo_benchmarks/../../scripts/train_loo.py:179: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[labels_key] = adata.obs[labels_key].astype("category")


================================================== Slide: crc_242, Holdout Celltype: Endothelial ==================================================


INFO: Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
2026-07-25 15:17:06 | [INFO] Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /data2/a330d/data/ood/trained/crc_242/Endothelial/cellina-W_1/model.pt already downloaded            


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


...

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


In [11]:
results_csv_name = f'../../results/homotypic_{DATASET_NAME}_DEG_{n_deg}.csv'
df_results = pd.DataFrame(results)

# If file exists, append results to existing csv, otherwise create new csv
if os.path.exists(results_csv_name):
    df_results.to_csv(f"{results_csv_name}", index=False, mode='a', header=False)
else:
    df_results.to_csv(f"{results_csv_name}", index=False)

In [12]:
df_results

,dataset_name,sid,control_domain,target_domain,n_deg,model_name,holdout_celltype,neighbor_celltype,spearman,pearson,...,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca_log,edistance_pca,rmse,mse_lfc,top_n_perturb
0,crc,crc_242,REF,CRC,50,cellina,Endothelial,Endothelial,0.667995,0.901085,...,0.20,1.00,0.477386,73.429128,75.184365,7.596127,206.451138,3280.596087,2.603477,200
1,crc,crc_242,REF,CRC,50,cellina,Endothelial,Epithelial,0.724370,0.908582,...,0.32,1.00,0.560171,71.825963,72.458765,7.798729,180.671357,2857.147895,1.645511,200
2,crc,crc_242,REF,CRC,50,cellina,Endothelial,Fibroblast,0.729844,0.913888,...,0.24,0.98,0.478414,74.618834,75.347412,7.098796,195.235628,3252.737454,2.625222,200
3,crc,crc_242,REF,CRC,50,cellina,Endothelial,Myeloid,0.695558,0.907823,...,0.20,0.98,0.484125,74.328062,74.729216,6.979168,208.700046,3275.499433,2.655747,200
4,crc,crc_242,REF,CRC,50,cellina,Endothelial,T_cell,0.661465,0.859025,...,0.14,0.92,0.424532,77.366103,77.955572,7.026154,259.930240,3507.238178,3.429488,200
5,crc,crc_242,REF,CRC,50,cellina,Epithelial,Endothelial,0.593950,0.660614,...,0.24,0.86,0.711789,63.052743,66.180307,11.609678,344.339943,79352.539939,10.608048,200
6,crc,crc_242,REF,CRC,50,cellina,Epithelial,Epithelial,0.538631,0.619872,...,0.26,0.84,0.477307,64.231626,66.273926,11.740024,365.964052,79958.736486,11.172790,200
7,crc,crc_242,REF,CRC,50,cellina,Epithelial,Fibroblast,0.655318,0.612078,...,0.18,0.84,0.325497,67.711334,70.193758,12.073137,454.864593,85743.372190,12.213021,200
8,crc,crc_242,REF,CRC,50,cellina,Epithelial,Myeloid,0.544010,0.591331,...,0.24,0.80,0.743836,64.776802,67.499811,11.812360,437.653139,84687.751639,12.534313,200
9,crc,crc_242,REF,CRC,50,cellina,Epithelial,T_cell,0.571381,0.608817,...,0.24,0.82,0.101369,65.029943,67.459797,11.892845,441.980725,84616.681556,12.938793,200


# Plot

In [19]:
results_csv_name = f'../../results/homotypic_{DATASET_NAME}_DEG_{n_deg}.csv'
df_results = pd.read_csv(results_csv_name)

In [20]:
df_results

,dataset_name,sid,control_domain,target_domain,n_deg,model_name,holdout_celltype,neighbor_celltype,spearman,pearson,...,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca_log,edistance_pca,rmse,mse_lfc,top_n_perturb
0,crc,crc_231,REF,CRC,50,cellina,Endothelial,Endothelial,0.787179,0.962958,...,0.50,1.00,0.893851,94.155707,100.183098,9.482941,168.310586,4921.682374,1.469765,200
1,crc,crc_231,REF,CRC,50,cellina,Endothelial,Epithelial,0.795822,0.963336,...,0.48,1.00,0.978641,94.152887,99.830463,9.874966,168.463822,4967.134430,1.442363,200
2,crc,crc_231,REF,CRC,50,cellina,Endothelial,Fibroblast,0.786699,0.963147,...,0.50,1.00,0.978641,94.735021,100.642866,9.305225,170.885211,4930.385945,1.459969,200
3,crc,crc_231,REF,CRC,50,cellina,Endothelial,Myeloid,0.790348,0.963019,...,0.50,1.00,0.893851,94.965394,101.095212,9.333343,180.302141,4923.143302,1.468349,200
4,crc,crc_231,REF,CRC,50,cellina,Endothelial,T_cell,0.785354,0.962808,...,0.50,1.00,0.956300,94.582519,99.687829,9.147322,167.989149,4920.389849,1.475917,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,crc,crc_242,REF,CRC,50,cellina,T_cell,Endothelial,0.686819,0.847746,...,0.26,0.96,0.721306,85.512672,89.837259,5.622622,167.907434,19113.200271,1.149575,200
96,crc,crc_242,REF,CRC,50,cellina,T_cell,Epithelial,0.831068,0.906955,...,0.40,0.98,0.726460,85.591015,90.861743,5.673579,141.412102,19084.817428,0.738427,200
97,crc,crc_242,REF,CRC,50,cellina,T_cell,Fibroblast,0.786603,0.852291,...,0.30,0.94,0.704525,87.523322,92.047502,4.780962,144.418053,19510.167270,1.249324,200
98,crc,crc_242,REF,CRC,50,cellina,T_cell,Myeloid,0.778343,0.869067,...,0.32,0.96,0.683587,85.787250,90.770856,5.060666,145.357992,19351.404182,1.174955,200


In [22]:
import pandas as pd

# Group by holdout x neighbor cell type, aggregate pearson
summary = (
    df_results.groupby(['holdout_celltype', 'neighbor_celltype'])['pearson']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)

# Format as "mean ± std" strings
summary['mean_std'] = summary.apply(
    lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1
)

# Pivot into k x k table: rows = holdout, cols = neighbor
table = summary.pivot(index='holdout_celltype', columns='neighbor_celltype', values='mean_std')

print(table)

neighbor_celltype  Endothelial   Epithelial   Fibroblast      Myeloid  \
holdout_celltype                                                        
Endothelial        0.85 ± 0.10  0.87 ± 0.08  0.88 ± 0.07  0.87 ± 0.08   
Epithelial         0.31 ± 0.49  0.26 ± 0.63  0.52 ± 0.22  0.53 ± 0.28   
Fibroblast         0.46 ± 0.07  0.47 ± 0.24  0.68 ± 0.20  0.66 ± 0.21   
Myeloid            0.78 ± 0.10  0.87 ± 0.04  0.83 ± 0.09  0.81 ± 0.11   
T_cell             0.89 ± 0.10  0.92 ± 0.08  0.90 ± 0.09  0.90 ± 0.09   

neighbor_celltype       T_cell  
holdout_celltype                
Endothelial        0.82 ± 0.11  
Epithelial         0.23 ± 0.54  
Fibroblast         0.34 ± 0.32  
Myeloid            0.77 ± 0.09  
T_cell             0.86 ± 0.12  


In [17]:
summary = (
    df_results.groupby(['holdout_celltype', 'neighbor_celltype'])['pearson']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)

# Which cells have too few samples?
print(summary[summary['count'] <= 1])

   holdout_celltype neighbor_celltype      mean  std  count
0       Endothelial       Endothelial  0.901085  NaN      1
1       Endothelial        Epithelial  0.908582  NaN      1
2       Endothelial        Fibroblast  0.913888  NaN      1
3       Endothelial           Myeloid  0.907823  NaN      1
4       Endothelial            T_cell  0.859025  NaN      1
5        Epithelial       Endothelial  0.660614  NaN      1
6        Epithelial        Epithelial  0.619872  NaN      1
7        Epithelial        Fibroblast  0.612078  NaN      1
8        Epithelial           Myeloid  0.591331  NaN      1
9        Epithelial            T_cell  0.608817  NaN      1
10       Fibroblast       Endothelial  0.545852  NaN      1
11       Fibroblast        Epithelial  0.830909  NaN      1
12       Fibroblast        Fibroblast  0.849930  NaN      1
13       Fibroblast           Myeloid  0.890600  NaN      1
14       Fibroblast            T_cell  0.780668  NaN      1
15          Myeloid       Endothelial  0

In [23]:
from scipy.stats import kruskal
import pandas as pd

results = []
for holdout, group in df_results.groupby('holdout_celltype'):
    # collect pearson values per neighbor_celltype as separate samples
    samples = [g['pearson'].values for _, g in group.groupby('neighbor_celltype')]
    # kruskal needs at least 2 groups with >=1 obs each, ideally more per group
    stat, p = kruskal(*samples)
    results.append({'holdout_celltype': holdout, 'H_stat': stat, 'p_value': p, 
                     'n_neighbor_groups': len(samples),
                     'n_total': sum(len(s) for s in samples)})

kw_results = pd.DataFrame(results).sort_values('p_value')
print(kw_results)

  holdout_celltype    H_stat   p_value  n_neighbor_groups  n_total
2       Fibroblast  5.606792  0.230501                  5       20
3          Myeloid  4.459623  0.347362                  5       20
4           T_cell  3.169057  0.529943                  5       20
0      Endothelial  2.337358  0.673976                  5       20
1       Epithelial  0.975094  0.913549                  5       20


In [24]:
from statsmodels.stats.multitest import multipletests
kw_results['p_adj'] = multipletests(kw_results['p_value'], method='fdr_bh')[1]

In [25]:
kw_results

,holdout_celltype,H_stat,p_value,n_neighbor_groups,n_total,p_adj
2,Fibroblast,5.606792,0.230501,5,20,0.842470
3,Myeloid,4.459623,0.347362,5,20,0.842470
4,T_cell,3.169057,0.529943,5,20,0.842470
0,Endothelial,2.337358,0.673976,5,20,0.842470
1,Epithelial,0.975094,0.913549,5,20,0.913549
